# Inventaire des genres par artiste

Ce notebook compte le nombre d'artistes par genre.

- **Données :** `data/genres_1000_cleaned.json`. Ce sont les genres de Metal Archives pour les 1 000 artistes les plus fournis en chansons. 604 artistes ont un genre. Les autres sont à `None`.
- on retire le suffixe « Metal » (Heavy Metal devient Heavy). On garde seulement le premier genre de chaque artiste.


In [ ]:
import json
import matplotlib.pyplot as plt
from collections import Counter
import pandas as pd
import torch
from transformers import GPT2Tokenizer, GPT2Model
from google.colab import drive
import pandas as pd



In [ ]:

drive.mount("/content/drive")
chansons_en = pd.read_csv("/content/drive/MyDrive/metal_mining/chansons_en_13_classes.csv")

print(len(chansons_en), "chansons")
print(chansons_en["genre"].nunique(), "genres")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
32486 chansons
13 genres


In [ ]:
def embed(model, tokenizer, paroles):
    entree = tokenizer(paroles, return_tensors="pt", truncation=True, max_length=1024).to(appareil)
    with torch.no_grad():
        sortie = model(**entree)
    return sortie.last_hidden_state.cpu().numpy()  # (1, n_tokens, 768)

In [ ]:
import time

tokenizer = GPT2Tokenizer.from_pretrained('lucone83/deep-metal')
model = GPT2Model.from_pretrained('lucone83/deep-metal')


appareil = "cuda" if torch.cuda.is_available() else "cpu"
print("appareil :", appareil)
model = model.to(appareil)


echantillon = chansons_en.sample(200, random_state=42)

debut = time.time()
matrices, nb_sautees = [], 0
for paroles in echantillon["paroles"]:
    try:
        matrices.append(embed(model=model, tokenizer=tokenizer, paroles=paroles))
    except IndexError:
        nb_sautees += 1
duree = time.time() - debut

print(round(duree), "s pour 200 chansons |", nb_sautees, "sautées")
print("estimation pour tout :", round(duree / 200 * len(chansons_en) / 3600, 1), "h")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2Model LOAD REPORT from: lucone83/deep-metal
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


appareil : cuda
1 s pour 200 chansons | 0 sautées
estimation pour tout : 0.0 h


In [ ]:
nb_tokens = chansons_en["paroles"].map(lambda texte: len(tokenizer(texte)["input_ids"]))
chansons_en["tronquee"] = nb_tokens > 1024

print(chansons_en["tronquee"].sum(), "chansons de plus de 1024 tokens sur", len(chansons_en))
print(round(chansons_en["tronquee"].mean() * 100, 2), "%")
print(chansons_en.groupby("genre")["tronquee"].mean().round(4))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1225 > 1024). Running this sequence through the model will result in indexing errors


67 chansons de plus de 1024 tokens sur 32486
0.21 %
genre
Black            0.0023
Death            0.0009
Doom             0.0000
Folk             0.0163
Gothic           0.0000
Grindcore        0.0000
Heavy            0.0010
Melodic Death    0.0007
Metalcore        0.0000
Power            0.0016
Progressive      0.0074
Symphonic        0.0109
Thrash           0.0008
Name: tronquee, dtype: float64


In [ ]:
import torch
print(torch.zeros(1).to("cuda"))

tensor([0.], device='cuda:0')


In [ ]:
import numpy as np
from tqdm.auto import tqdm


agregations = {"moyenne": [], "mediane": [], "last_token": [], "variance": []}
index_ok = []
index_ignores = []

for index, paroles in tqdm(chansons_en["paroles"].items(), total=len(chansons_en)):
    if not paroles.strip():
        index_ignores.append(index)
        continue

    matrice = embed(model=model, tokenizer=tokenizer, paroles=paroles)[0]  # (n_tokens, 768)

    agregations["moyenne"].append(matrice.mean(axis=0))
    agregations["mediane"].append(np.median(matrice, axis=0))
    agregations["last_token"].append(matrice[-1])
    agregations["variance"].append(matrice.var(axis=0, ddof=0))
    index_ok.append(index)

X_par_methode = {methode: np.vstack(vecteurs).astype(np.float32)
                 for methode, vecteurs in agregations.items()}
X_par_methode["moyenne_variance"] = np.hstack([X_par_methode["moyenne"], X_par_methode["variance"]])

print(len(index_ok), "chansons traitées |", len(index_ignores), "ignorées")
for methode, X in X_par_methode.items():
    print(methode, X.shape, "NaN :", int(np.isnan(X).sum()))

  0%|          | 0/32486 [00:00<?, ?it/s]

32486 chansons traitées | 0 ignorées
moyenne (32486, 768) NaN : 0
mediane (32486, 768) NaN : 0
last_token (32486, 768) NaN : 0
variance (32486, 768) NaN : 0
moyenne_variance (32486, 1536) NaN : 0


In [ ]:
dossier = "/content/drive/MyDrive/metal_mining/"

np.savez(dossier + "X_par_methode_13_classes.npz", **X_par_methode)
np.save(dossier + "index_ok_13_classes.npy", np.array(index_ok))